# 01 · Esplorazione delle sorgenti di un flusso ETL

**Demo «Python e VS Code nei flussi ETL»**

Prima di costruire un flusso dobbiamo conoscere i dati che dovrà trattare. In questo notebook apriamo i file sorgente, li osserviamo e cerchiamo i problemi che contengono. È lo stesso lavoro che in uno strumento grafico si fa con le funzioni di anteprima dei dati: qui lo facciamo scrivendo poche righe di codice e guardando subito il risultato.

**Che cosa faremo, passo per passo**

1. Prepariamo l'ambiente di lavoro.
2. Guardiamo quali file abbiamo a disposizione.
3. Leggiamo il file dei movimenti così com'è.
4. Andiamo a caccia dei problemi nei dati.
5. Mettiamo a punto una trasformazione: prima «così com'è», poi nel modo idiomatico.
6. Vediamo come la stessa logica vive nel progetto, sotto forma di funzioni riutilizzabili.
7. Interroghiamo il database di destinazione, dopo aver eseguito il flusso.

## Assunzioni

Elenchiamo subito ciò che diamo per vero in questo notebook. Se una di queste assunzioni non vale, alcune celle potrebbero non funzionare oppure dare risultati diversi da quelli descritti.

1. **Ambiente.** Il notebook viene eseguito in VS Code usando l'ambiente conda `etl_demo`, creato come indicato nel file `README.md`. In alto a destra, nel selettore del *kernel*, deve comparire `etl_demo`.
2. **Posizione.** Il notebook si trova nella cartella `notebook` del progetto `demo_etl_python` e non è stato spostato altrove.
3. **Dati.** La cartella `dati/input` contiene i file prodotti dallo script `strumenti/genera_dati.py`. Sono dati sintetici, cioè inventati: nessun cliente, conto o movimento è reale.
4. **Formato dei file.** I file CSV usano il punto e virgola come separatore di colonna e la codifica UTF-8. Nel file dei movimenti le date sono scritte nel formato giorno/mese/anno (ad esempio `17/11/2026`) e gli importi nel formato italiano, con il punto per le migliaia e la virgola per i decimali (ad esempio `1.234,56`).
5. **Giornata di riferimento.** Lavoriamo sul file dei movimenti del 17 novembre 2026.
6. **Versione di pandas.** Il notebook è stato provato con pandas 3.0.
7. **Ultima sezione.** La sezione 7 richiede che il flusso sia stato eseguito almeno una volta per il 17 novembre 2026: è il flusso a creare il database che andremo a interrogare.

## Le parole che useremo

| Termine | Significato |
|---|---|
| **ETL** | Sigla di *Extract, Transform, Load*: estrarre i dati dalle sorgenti, trasformarli, caricarli nella destinazione. |
| **Sorgente** | Il luogo da cui i dati arrivano. Qui: file CSV e un file Excel. |
| **Destinazione** | Il luogo in cui i dati trasformati vengono scritti. Qui: un database SQLite. |
| **CSV** | File di testo in cui ogni riga è un record e le colonne sono separate da un carattere, nel nostro caso il punto e virgola. |
| **Notebook** | Un documento fatto di *celle*: alcune contengono testo (come questa), altre contengono codice che possiamo eseguire una alla volta, vedendo subito il risultato sotto la cella. |
| **Kernel** | Il programma Python che esegue le celle del notebook. Scegliere il kernel significa scegliere l'ambiente conda da usare. |
| **pandas** | La libreria Python più diffusa per lavorare con tabelle di dati. |
| **DataFrame** | La tabella di pandas: righe e colonne, come un foglio di calcolo o una tabella di database. |
| **Scarto** | Una riga che non supera un controllo e che quindi non viene caricata. Non la buttiamo via: la mettiamo da parte insieme al motivo. |
| **Data di elaborazione** | Il giorno a cui si riferiscono i dati che stiamo caricando. |

## 1 · Prepariamo l'ambiente di lavoro

Importiamo le librerie e controlliamo con quale Python stiamo lavorando.

In [ ]:
# Importiamo gli strumenti che ci servono.
import sys                  # informazioni sull'interprete Python in uso
import time                 # per misurare quanto tempo impiega un'operazione
from pathlib import Path    # per indicare cartelle e file nello stesso modo su Windows, Linux e macOS

import pandas as pd         # la libreria per le tabelle di dati ("pd" è l'abbreviazione che usano tutti)

print("Versione di Python:", sys.version.split()[0])
print("Versione di pandas:", pd.__version__)
print("Interprete in uso: ", sys.executable)

**Che cosa controllare.** L'ultima riga stampata mostra il percorso dell'*interprete*, cioè del programma `python` che sta eseguendo le celle. Il percorso deve contenere il nome dell'ambiente conda, `etl_demo`. Se non è così, stiamo usando un Python diverso da quello preparato per la demo: conviene fermarsi e scegliere il kernel corretto con il selettore in alto a destra.

Ora individuiamo la cartella principale del progetto e leggiamo il file dei parametri. Usiamo la stessa funzione che usa il flusso vero e proprio: in questo modo notebook e flusso leggono i file dalle stesse cartelle.

In [ ]:
# Individuiamo la cartella principale del progetto.
# Un notebook aperto in VS Code lavora di norma nella cartella in cui si trova (qui: notebook).
# In quel caso la cartella del progetto è quella immediatamente superiore.
RADICE_PROGETTO = Path.cwd()
if not (RADICE_PROGETTO / "config.toml").exists():
    RADICE_PROGETTO = RADICE_PROGETTO.parent

# Aggiungiamo la cartella del progetto all'elenco dei posti in cui Python cerca i moduli:
# così anche da qui possiamo importare il nostro pacchetto "etl".
if str(RADICE_PROGETTO) not in sys.path:
    sys.path.insert(0, str(RADICE_PROGETTO))

from etl.configurazione import carica_configurazione

config = carica_configurazione()                        # legge config.toml
CARTELLA_INPUT = config["percorsi"]["cartella_input"]   # la cartella dei file sorgente

print("Cartella del progetto:     ", RADICE_PROGETTO)
print("Cartella dei file di input:", CARTELLA_INPUT)

## 2 · Quali file abbiamo a disposizione

Elenchiamo il contenuto della cartella di input, con la dimensione di ogni file.

In [ ]:
# Scorriamo i file della cartella di input in ordine alfabetico
for file in sorted(CARTELLA_INPUT.iterdir()):
    dimensione_kb = file.stat().st_size / 1024          # la dimensione è in byte: la trasformiamo in kilobyte
    print(f"{file.name:<26} {dimensione_kb:>10.1f} KB")

| File | Contenuto | Ruolo nel flusso |
|---|---|---|
| `movimenti_AAAAMMGG.csv` | I movimenti di conto corrente di una giornata. Ne arriva uno al giorno. | È la sorgente principale: il file da controllare e caricare. |
| `conti.csv` | L'anagrafica dei conti: per ogni conto il cliente, la filiale, il tipo e lo stato. | Tabella di decodifica: aggiunge ai movimenti cliente e filiale. |
| `clienti.csv` | L'anagrafica dei clienti, con il segmento commerciale. | Tabella di decodifica: aggiunge ai movimenti il segmento. |
| `cambi.xlsx` | I cambi giornalieri contro euro, in un file Excel. | Serve a convertire in euro i movimenti in altre divise. |

## 3 · Leggiamo il file dei movimenti così com'è

Leggiamo il file del 17 novembre 2026 chiedendo a pandas di **non interpretare nulla**: tutte le colonne vengono lette come semplice testo. Fra poco vedremo perché questa prudenza è importante.

In [ ]:
FILE_MOVIMENTI = CARTELLA_INPUT / "movimenti_20261117.csv"

grezzi = pd.read_csv(
    FILE_MOVIMENTI,
    sep=";",                  # il separatore di colonna è il punto e virgola
    dtype=str,                # leggiamo TUTTE le colonne come testo
    keep_default_na=False,    # una cella vuota resta testo vuoto "", non diventa "valore mancante"
    encoding="utf-8",
)

print("Righe lette:", len(grezzi), "- Colonne:", len(grezzi.columns))
grezzi.head(10)               # head(10) mostra le prime dieci righe

**In VS Code.** Sotto la tabella compare il pulsante **Open 'grezzi' in Data Wrangler**. Data Wrangler è un'estensione di VS Code che apre la tabella in una griglia interattiva: possiamo ordinare e filtrare le colonne e, per ciascuna, vedere quanti valori distinti e quanti valori mancanti contiene. È lo strumento più vicino all'anteprima dei dati di un prodotto grafico. Lo stesso strumento si apre anche senza scrivere codice: clic destro su un file CSV nell'elenco dei file e poi **Open in Data Wrangler**.

### Perché leggere tutto come testo

Potremmo chiedere a pandas di convertire subito gli importi, indicandogli che la virgola separa i decimali e il punto le migliaia. Sembra comodo, ma guardiamo che cosa succede con un file di due sole righe, nella seconda delle quali qualcuno ha usato per errore il punto come separatore dei decimali.

In [ ]:
import io   # ci permette di trattare un testo come se fosse un file

# Un "file" di due righe: nella riga A l'importo è scritto all'italiana (1.234,56),
# nella riga B è scritto con il punto decimale (1234.56). I due importi sono uguali.
file_di_esempio = io.StringIO("id;importo\nA;1.234,56\nB;1234.56\n")

lettura_automatica = pd.read_csv(file_di_esempio, sep=";", decimal=",", thousands=".")
lettura_automatica

L'importo della riga B è diventato **123456,00**: cento volte il valore corretto, e senza alcun avviso. pandas ha fatto ciò che gli abbiamo chiesto, cioè ha considerato ogni punto come separatore delle migliaia e lo ha eliminato.

Da qui la regola che seguiamo in tutto il flusso:

> **Prima controlliamo, poi convertiamo.** Leggiamo i dati come testo, verifichiamo che ogni valore rispetti il formato atteso e convertiamo solo i valori che superano il controllo. Gli altri diventano scarti.

## 4 · A caccia dei problemi

Esaminiamo il file con cinque controlli. Per ognuno contiamo le righe coinvolte e guardiamo qualche esempio.

### 4.1 · Date non valide

Proviamo a convertire la colonna delle date dal testo al tipo «data». Indichiamo il formato atteso con `%d/%m/%Y`, che significa giorno, mese e anno di quattro cifre separati da barre.

In [ ]:
# Con errors="coerce" le date che non rispettano il formato non provocano un errore:
# diventano NaT ("Not a Time", cioè "data mancante") e possiamo contarle.
date_convertite = pd.to_datetime(grezzi["data_operazione"], format="%d/%m/%Y", errors="coerce")

data_non_valida = date_convertite.isna()    # una colonna di Vero/Falso: Vero dove la data non è valida

print("Righe con la data non valida:", data_non_valida.sum())

# value_counts conta quante volte compare ciascun valore: vediamo quali sono le date sbagliate
grezzi.loc[data_non_valida, "data_operazione"].value_counts()

Troviamo date impossibili (il 31 febbraio, la data fatta di soli zeri), date vuote e date scritte in un formato diverso da quello concordato.

### 4.2 · Importi non validi

Per gli importi descriviamo il formato atteso con una **espressione regolare**, cioè un modello che dice come deve essere fatto un testo. La nostra si legge così:

| Pezzo del modello | Significato |
|---|---|
| `-?` | un segno meno facoltativo |
| `\d{1,3}(\.\d{3})+` | da una a tre cifre, seguite da uno o più gruppi formati da un punto e tre cifre (le migliaia) |
| `\|\d+` | oppure, in alternativa, una semplice sequenza di cifre senza punti |
| `,\d{2}` | la virgola seguita da esattamente due cifre decimali |

Il metodo `str.fullmatch` controlla che **tutto** il testo, dall'inizio alla fine, rispetti il modello.

In [ ]:
FORMATO_IMPORTO = r"-?(\d{1,3}(\.\d{3})+|\d+),\d{2}"

importo_valido = grezzi["importo"].str.fullmatch(FORMATO_IMPORTO)   # Vero dove l'importo rispetta il formato

# Il simbolo ~ significa "non": ~importo_valido è Vero dove l'importo NON è valido
print("Righe con l'importo non valido:", (~importo_valido).sum())
grezzi.loc[~importo_valido, "importo"].value_counts()

Fra gli importi rifiutati c'è proprio `1234.56`, il caso dell'esempio precedente: questa volta lo intercettiamo invece di trasformarlo in un valore sbagliato.

### 4.3 · Codici scritti in modi diversi

Contiamo quante volte compare ciascun valore della colonna `divisa`.

In [ ]:
grezzi["divisa"].value_counts()

La stessa divisa compare scritta in più modi: `EUR`, `eur`, ` eur` con uno spazio davanti. Per una persona sono la stessa cosa, per un programma sono tre valori diversi. Questo non è un motivo di scarto: basta **ripulire** il codice togliendo gli spazi ai lati (`strip`) e portando tutto in maiuscolo (`upper`).

In [ ]:
divisa_pulita = grezzi["divisa"].str.strip().str.upper()
divisa_pulita.value_counts()

Ora le divise sono cinque. Notiamo la presenza di `JPY`, lo yen giapponese: ci torneremo al punto 4.6.

### 4.4 · Righe ripetute

Ogni movimento ha un identificativo, `id_movimento`, che deve essere unico. Verifichiamo se qualche identificativo compare più di una volta.

In [ ]:
ripetuto = grezzi["id_movimento"].duplicated()   # Vero dalla seconda comparsa in poi dello stesso identificativo
print("Righe ripetute:", ripetuto.sum())

# Guardiamo un caso concreto: prendiamo il primo identificativo ripetuto e mostriamo tutte le sue righe
esempio = grezzi.loc[ripetuto, "id_movimento"].iloc[0]
grezzi[grezzi["id_movimento"] == esempio]

Le due righe sono identiche: lo stesso movimento è stato inviato due volte. Nel flusso terremo la prima e scarteremo la ripetizione, altrimenti quel movimento verrebbe contato doppio.

### 4.5 · Conti che non esistono in anagrafica

Leggiamo l'anagrafica dei conti e verifichiamo che il conto di ogni movimento vi sia presente. Prima del confronto ripuliamo il codice del conto, come abbiamo fatto per la divisa.

In [ ]:
conti = pd.read_csv(CARTELLA_INPUT / "conti.csv", sep=";", dtype=str)
print("Conti in anagrafica:", len(conti))

conto_pulito = grezzi["id_conto"].str.strip().str.upper()
conto_presente = conto_pulito.isin(conti["id_conto"])     # Vero se il conto del movimento esiste in anagrafica

print("Movimenti su conti che non esistono in anagrafica:", (~conto_presente).sum())
grezzi[~conto_presente].head()

Un movimento su un conto sconosciuto non può essere attribuito a nessun cliente e a nessuna filiale: verrà scartato.

### 4.6 · Divise per le quali manca il cambio

Leggiamo la tabella dei cambi dal file Excel e guardiamo i cambi del 17 novembre. Il cambio indica quante unità di divisa servono per comprare 1 euro.

In [ ]:
cambi = pd.read_excel(CARTELLA_INPUT / "cambi.xlsx", sheet_name="cambi")
cambi[cambi["data"] == "2026-11-17"]

In [ ]:
# Confrontiamo due insiemi: le divise usate nei movimenti e le divise per cui abbiamo un cambio.
# La sottrazione fra insiemi restituisce gli elementi del primo che non stanno nel secondo.
divise_dei_movimenti = set(divisa_pulita)
divise_con_cambio = set(cambi["divisa"])
divise_senza_cambio = divise_dei_movimenti - divise_con_cambio

print("Divise usate nei movimenti ma assenti dalla tabella dei cambi:", divise_senza_cambio)
print("Movimenti coinvolti:", divisa_pulita.isin(divise_senza_cambio).sum())

I movimenti in yen non possono essere convertiti in euro, perché la tabella dei cambi non contiene lo yen: verranno scartati.

### Riepilogo dei problemi trovati

Raccogliamo in una tabella ciò che abbiamo scoperto e la decisione presa per ciascun problema.

In [ ]:
riepilogo_problemi = pd.DataFrame(
    {
        "problema": [
            "Data non valida",
            "Importo non valido",
            "Riga ripetuta",
            "Conto inesistente",
            "Cambio mancante",
            "Codice scritto in modo diverso",
        ],
        "righe": [
            int(data_non_valida.sum()),
            int((~importo_valido).sum()),
            int(ripetuto.sum()),
            int((~conto_presente).sum()),
            int(divisa_pulita.isin(divise_senza_cambio).sum()),
            int((grezzi["divisa"] != divisa_pulita).sum()),
        ],
        "decisione": ["scartare", "scartare", "scartare", "scartare", "scartare", "ripulire e tenere"],
    }
)
riepilogo_problemi

## 5 · Mettiamo a punto una trasformazione

Prendiamo la trasformazione più tipica di questo file: portare l'importo dal testo `1.234,56` al numero `1234.56`.

La scriviamo in due modi. Prima la versione **«così com'è»** (in inglese *as-is*), cioè quella che viene spontanea a chi arriva da altri linguaggi: un ciclo che elabora una riga alla volta. Poi la versione **idiomatica**, cioè scritta nel modo proprio di pandas: una sola istruzione che lavora sull'intera colonna.

### Così com'è: una riga alla volta

In [ ]:
# Lavoriamo solo sulle righe con l'importo nel formato corretto.
# copy() crea una tabella indipendente, che possiamo modificare senza toccare "grezzi".
validi = grezzi[importo_valido].copy()
validi["importo_numerico"] = 0.0              # prepariamo la colonna che conterrà il risultato

inizio = time.perf_counter()                  # facciamo partire il cronometro

# iterrows restituisce, una alla volta, l'etichetta e il contenuto di ogni riga
for indice, riga in validi.iterrows():
    testo = riga["importo"]                   # ad esempio "1.234,56"
    testo = testo.replace(".", "")            # togliamo il punto delle migliaia: "1234,56"
    testo = testo.replace(",", ".")           # la virgola decimale diventa punto: "1234.56"
    validi.loc[indice, "importo_numerico"] = float(testo)   # scriviamo il numero nella riga corrispondente

tempo_ciclo = time.perf_counter() - inizio    # fermiamo il cronometro

print(f"Righe convertite: {len(validi)} - Tempo impiegato: {tempo_ciclo:.2f} secondi")

### Idiomatico: l'intera colonna in una sola istruzione

In [ ]:
inizio = time.perf_counter()

importo_numerico = (
    validi["importo"]
    .str.replace(".", "", regex=False)    # togliamo il punto delle migliaia in tutta la colonna
    .str.replace(",", ".", regex=False)   # la virgola decimale diventa punto in tutta la colonna
    .astype(float)                        # convertiamo tutta la colonna da testo a numero
)

tempo_colonna = time.perf_counter() - inizio

print(f"Righe convertite: {len(importo_numerico)} - Tempo impiegato: {tempo_colonna:.4f} secondi")
print(f"La versione idiomatica è circa {tempo_ciclo / tempo_colonna:.0f} volte più veloce.")
print("I due risultati coincidono:", importo_numerico.equals(validi["importo_numerico"]))

I due risultati sono identici, ma la versione idiomatica è più breve, si legge come una ricetta in tre passi ed è più veloce di centinaia di volte. Su ventimila righe la differenza è di pochi secondi; su venti milioni di righe è la differenza fra un flusso che termina in pochi secondi e uno che richiede decine di minuti.

> **Regola pratica.** Quando in un programma pandas compare un ciclo `for` che scorre le righe di una tabella, quasi sempre esiste un'istruzione che fa lo stesso lavoro sull'intera colonna.

## 6 · Dal notebook al progetto

Il notebook è lo strumento giusto per **capire**: proviamo, guardiamo, correggiamo. Non è lo strumento giusto per **mandare in produzione** un flusso, perché le celle si possono eseguire in qualunque ordine e il codice non è facilmente riutilizzabile né verificabile in modo automatico.

Per questo, una volta chiarite le regole, le abbiamo riscritte nel progetto sotto forma di **funzioni**, nel file `etl/trasformazione.py`. Una funzione è un pezzo di codice con un nome, che riceve dei dati in ingresso e restituisce un risultato: si può richiamare da più punti e si può provare con dei test automatici.

Richiamiamo quelle funzioni da qui, sullo stesso file. Ognuna restituisce due tabelle: le righe valide, che proseguono, e gli scarti.

In [ ]:
from etl.estrazione import leggi_movimenti
from etl.trasformazione import controlla_formati, togli_duplicati

grezzi_dal_progetto = leggi_movimenti(FILE_MOVIMENTI)                 # lettura come testo, con il numero di riga
movimenti, scarti_formati = controlla_formati(grezzi_dal_progetto)    # controllo e conversione di date e importi
movimenti, scarti_duplicati = togli_duplicati(movimenti)              # eliminazione delle righe ripetute

print("Righe lette:              ", len(grezzi_dal_progetto))
print("Scartate per il formato:  ", len(scarti_formati))
print("Scartate perché ripetute: ", len(scarti_duplicati))
print("Righe che proseguono:     ", len(movimenti))

I conteggi coincidono con quelli trovati a mano nella sezione 4. Guardiamo i tipi delle colonne dopo la trasformazione: la data è diventata una vera data (`datetime64`) e l'importo un vero numero (`float64`).

In [ ]:
movimenti.dtypes     # dtypes elenca il tipo di dato di ogni colonna

In [ ]:
# Degli scarti teniamo il numero della riga nel file di origine e il motivo
scarti_formati.head()

A questo punto conviene passare ai file del progetto, nell'ordine in cui il flusso li usa:

1. `esegui_etl.py`: il programma che si lancia;
2. `etl/flusso.py`: i passi del flusso, uno dopo l'altro;
3. `etl/trasformazione.py`: le regole viste in questo notebook.

Il flusso si lancia dal terminale di VS Code con il comando:

```
python esegui_etl.py --data 2026-11-17
```

## 7 · Dopo l'esecuzione: interroghiamo la destinazione

Questa sezione va eseguita **dopo** aver lanciato il flusso almeno una volta.

La destinazione è un database **SQLite**: un database contenuto in un unico file, che non richiede alcun server. Python lo gestisce con il modulo standard `sqlite3`. Con pandas possiamo eseguire un'interrogazione scritta in **SQL**, il linguaggio dei database, e ricevere il risultato sotto forma di DataFrame.

In [ ]:
import sqlite3   # modulo standard di Python per i database SQLite

FILE_DATABASE = config["percorsi"]["file_database"]

# Se il database non esiste ancora ci fermiamo con un messaggio che spiega che cosa fare
if not FILE_DATABASE.exists():
    raise FileNotFoundError(
        "Il database non esiste ancora. Eseguire prima il flusso dal terminale di VS Code:\n"
        "    python esegui_etl.py --data 2026-11-17"
    )

connessione = sqlite3.connect(FILE_DATABASE)   # la "connessione" è il canale aperto verso il database
print("Collegati a:", FILE_DATABASE)

### Lo storico delle esecuzioni

Il flusso annota ogni esecuzione, riuscita o fallita, nella tabella `log_esecuzioni`.

In [ ]:
storico = pd.read_sql_query(
    "SELECT * FROM log_esecuzioni ORDER BY id_esecuzione",
    connessione,
    dtype_backend="numpy_nullable",   # così i numeri interi restano interi anche dove una cella è vuota
)
storico

### Quanti movimenti sono stati caricati, giorno per giorno

In [ ]:
pd.read_sql_query(
    '''
    SELECT data_elaborazione,
           COUNT(*)                   AS movimenti,
           ROUND(SUM(importo_eur), 2) AS saldo_eur,
           SUM(flag_rilevante)        AS movimenti_rilevanti
    FROM movimenti
    GROUP BY data_elaborazione
    ORDER BY data_elaborazione
    ''',
    connessione,
)

### Le cinque filiali con più movimenti nella giornata

Il punto interrogativo nel testo dell'interrogazione è un **segnaposto**: il valore, in questo caso la data, viene passato a parte con `params`. È la stessa tecnica che il flusso usa per scrivere nel database.

In [ ]:
pd.read_sql_query(
    '''
    SELECT codice_filiale,
           SUM(numero_movimenti)     AS movimenti,
           ROUND(SUM(totale_eur), 2) AS saldo_eur
    FROM riepilogo_giornaliero
    WHERE data_elaborazione = ?
    GROUP BY codice_filiale
    ORDER BY movimenti DESC
    LIMIT 5
    ''',
    connessione,
    params=("2026-11-17",),
)

### Le righe scartate

Gli scarti non finiscono nel database ma in un file CSV, uno per giornata. Ogni riga riporta il numero della riga nel file di origine, il motivo dello scarto e il contenuto originale, scritto esattamente come era nel file.

In [ ]:
FILE_SCARTI = config["percorsi"]["cartella_scarti"] / "scarti_movimenti_20261117.csv"
scarti = pd.read_csv(FILE_SCARTI, sep=";", dtype=str, keep_default_na=False)

print(scarti["motivo_scarto"].value_counts())
scarti.head(10)

In [ ]:
connessione.close()   # chiudiamo il collegamento al database quando abbiamo finito

## In sintesi

- Prima di costruire un flusso **guardiamo i dati**: il notebook e Data Wrangler sono gli strumenti per farlo.
- Leggiamo i file **come testo**, controlliamo il formato e convertiamo solo ciò che supera il controllo: una conversione automatica può sbagliare senza avvisare.
- Le righe sbagliate non si buttano e non si correggono a caso: diventano **scarti**, ciascuno con il proprio motivo.
- In pandas si lavora **per colonne**, non riga per riga: il codice è più breve e molto più veloce.
- Le regole messe a punto nel notebook diventano **funzioni** del progetto, riutilizzabili e verificabili con i test.

Il passo successivo è il notebook `02_grandi_volumi.ipynb`, dove vediamo che cosa fare quando le righe non sono ventimila ma diversi milioni.